# BEGINNNING

In [10]:
%load_ext autoreload
%autoreload 2

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'

import json
from thefuzz import fuzz, process
import re
import ujson
import logging
import inflect
from collections import Counter, defaultdict
from transformers import AutoTokenizer
import numpy as np
from sentence_transformers import SentenceTransformer, util
import evaluate
import spacy
import ast
import pandas as pd
# from vllm import LLM, SamplingParams
import torch
import matplotlib.pyplot as plt

from data_utils2 import *

# seqeval evaluation
seqeval = evaluate.load("seqeval")

# spacy tokenizer
nlp = spacy.blank("en")

# Create an engine object
p = inflect.engine()
# Set up logging configuration
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')


/nethome/cye73/conda_envs/bioner/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 10-03 21:13:00 [importing.py:53] Triton module has been replaced with a placeholder.
INFO 10-03 21:13:00 [__init__.py:239] Automatically detected platform cuda.
WARNING 10-03 21:13:01 [cuda.py:409] Detected different devices in the system: NVIDIA H100 NVL, NVIDIA A40, Tesla V100-PCIE-32GB, NVIDIA A40, NVIDIA A40. Please make sure to set `CUDA_DEVICE_ORDER=PCI_BUS_ID` to avoid unexpected behavior.


2025-10-03 21:13:02,507	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [ ]:
import openai
from ids import open_ai_api_key
openai.api_key = open_ai_api_key

In [ ]:
format_spans = "@@text##entity@@"
system_instructions = """You are a medical doctor who specializes in clinical trials and observational studies. 
You will act as an expert annotator of research articles abstracts provided to you. 
Another professional annotated those abstracts and your role is to verify the annotations and correct them if necessary.
"""
# You will return the strings from the article that match the tags provided. Do not convert words describing numbers into actual numbers, only return strings present in the abstract. 


In [ ]:
dataset_name = "mm_st21pv"
dataset_dir = "mm_st21pv"
tag_to_label = tag2label_st21pv
extraction_prompts = extraction_prompts_st21pv_v2

# dataset_name = "bc5cdr"
# dataset_dir = "bc5cdr"
# tag_to_label = tag2label_bc5cdr
# extraction_prompts = extraction_prompts_bc5cdr

# dataset_name = "ncbi_disease"
# dataset_dir = "ncbi_disease"
# tag_to_label = tag2label_ncbi
# extraction_prompts = extraction_prompts_ncbi

data_path = f"../data/{dataset_dir}/{dataset_name}.json"
with open(data_path, "r") as f:
    data = json.load(f)

for pmid, d in data.items() : 
    d['pmid'] = pmid
data_train = [data[pmid] for pmid in data if data[pmid]["split"] == "train"]

print(len(data), len(data_train))

# First load

In [ ]:
## LLM results from dataset corruption

data_folder = "mm_st21pv"
# data_folder = "mm_st21pv_BioLinkBERT_noise_50pc"

# model_version = "gpt-4o-2024-11-20_version=v4"
# model_version = "phi-4_version=v15"
# model_version = "Llama-3.1-8B-Instruct_version=v4"
# model_version = "Qwen2.5-14B-Instruct_version=v6"
# model_version = "Mistral-Nemo-Instruct-2407_version=v5"

## LLM results for annotations
# model_version = "gpt-4o-2024-11-20_dynamic=True_k=10_v11"
# model_version = "gpt-4o-2024-11-20_dynamic=True_k=10_testSet"
# model_version = "gpt-4o-2024-11-20_dynamic=True_k=20_version=v1"
# model_version = "phi-4_dynamic=True_k=5_version=v11"
# model_version = "phi-4_dynamic=True_k=5_testSet_v4"
# model_version = "Llama-3.1-8B-Instruct_dynamic=True_k=3_version=v1"
# model_version = "Llama-3.1-8B-Instruct_dynamic=True_k=1_testSet_v4"
# model_version = "Mistral-Nemo-Instruct-2407_dynamic=True_k=3_version=v1"
# model_version = "gpt-4o-2024-11-20_dynamic=True_k=5_version=v1"
# model_version = "gemma-3-12b-it_dynamic=True_k=5_version=v1"
model_version = "gemma-3-12b-it_dynamic=True_k=10_testSet_v4"
# model_version = "Mistral-Small-3.1-24B-Instruct-2503_dynamic=True_k=3_v1"
# model_version = "Mistral-Small-3.1-24B-Instruct-2503_dynamic=True_k=5_v2"
# model_version = "mistral-small-3.1-24b-instruct-2503-hf_dynamic=True_k=10_testSet_v4"

# data_folder = "bc5cdr"
# data_folder = "ncbi_disease"
# model_version = "Llama-3.1-8B-Instruct_dynamic=True_k=10_testSet_v2"
# model_version = "gemma-3-27b-it_dynamic=True_k=10_testSet_v2"
# model_version = "gpt-4o-2024-11-20_dynamic=True_k=10_testSet_v2"
# model_version = "phi-4_dynamic=True_k=10_testSet_v2"
# model_version = "mistral-small-3.1-24b-instruct-2503-hf_dynamic=True_k=10_testSet_v2"

raw_noisy_data_path = f"../llm/raw_results/{data_folder}/{model_version}.json" # Results from LLM inference
# raw_noisy_data_path = f"../raw_noisy_datasets/{data_folder}/{model_version}.json" # Results from dataset corruption
with open(raw_noisy_data_path, "r") as f:
    raw_noisy_data = json.load(f)
print("Number of raw noisy data :", len(raw_noisy_data))

noisy_data_path = f"../llm/results/{data_folder}/{model_version}.json" # Results from LLM inference
# noisy_data_path = f"../noisy_datasets/{data_folder}/{model_version}.json" # Results from dataset corruption
with open(noisy_data_path, "r") as f:
    llm_output = json.load(f)
with open(noisy_data_path, "r") as f:
    noisy_data = json.load(f)

print("Number of noisy data :", len(noisy_data))

# data_train_v1 = {pmid: data[pmid] for pmid in data if data[pmid]["split"] == "test"} # For LLM inference results
# data_train_v1 = {pmid: data[pmid] for pmid in data if data[pmid]["split"] == "train"} # For dataset corruption results
# total_n_true = sum([len(data[pmid]['spans']) for pmid in data_train_v1])
# noisy_data = {d['pmid']: d for d in noisy_data} # (not needed for results that were merged, they are already in a dict format)

noisy_data = {item["pmid"]: item for item in noisy_data} 
data_train_v1 = {pmid: data[pmid] for pmid in noisy_data}

total_n_true = 0
for pmid in noisy_data:
    true_spans = data_train_v1[pmid]['spans']
    true_spans = fuse_adjacent_tag(true_spans)
    total_n_true += len(true_spans)

# raw_noisy_data = {d['pmid']: d for d in raw_noisy_data} #

In [ ]:
# pmids_test = [pmid for pmid in data if data[pmid]["split"] == "test"]

# results = []
# nb_failure = 0
# for i, pmid in enumerate(pmids_test):
#     print("i :", i, "pmid :", pmid)
#     abstract, true_spans = data[pmid]["input"], data[pmid]["spans"]
#     if pmid not in raw_noisy_data:
#         continue
#     answer = raw_noisy_data[pmid]["llm_output"]

#     final_answer = extract_json_from_text(answer)

#     tagged_text = parse_answer(final_answer)

#     if tagged_text is None:
#         print(f"Skipping PMID {pmid} due to JSON parsing failure.")
#         nb_failure += 1
#         continue  # Skip this PMID for evaluation

#     pred_spans, reconstructed_text, count = gather_tagged_entities(
#         abstract, tagged_text["output"], tag_to_label
#     )
#     if reconstructed_text != abstract:
#         print(
#             f"Reconstructed text does not match original text for PMID {pmid}."
#         )
#         print("Length original text: ", len(abstract))
#         print("Length reconstructed text: ", len(reconstructed_text))
#         nb_reconstruction_mismatch += 1
#         continue

#     final_result = {
#         "pmid": pmid,
#         "tagged_text": tagged_text["output"].strip(),
#         "pred_spans": pred_spans,
#     }
#     results.append(final_result)
    
# dynamic = True
# k = 3
# eval_set = "test"
# version = "v4"
# llm_subname = "gemma-3-12b-it"
# file_name = f"{llm_subname}_dynamic={dynamic}_k={k}_{eval_set}Set_{version}"
# if file_name == model_version:
#     print("True")
#     with open(f"results/{dataset_dir}/{file_name}.json", "w") as f:
#         json.dump(results, f)

In [ ]:
final_noise = {}
nb_invalids_noise = set()
nb_invalids_empty = set()
nb_invalids_hallucination = set()
nb_untouched = set()
nb_corrects = 0
total_noise_count = 0
total_correct_count = 0
total_n_true = 0

full_results = []

error_analysis = defaultdict(int)

for pmid in noisy_data:
# for pmid in list(noisy_data.keys())[:2]:
# for pmid in ['25763772']:
    llm_annotation = noisy_data[pmid]["tagged_text"]
    abstract, true_spans = data[pmid]['input'], data_train_v1[pmid]['spans']
    pred_spans = noisy_data[pmid]['pred_spans']
    
    # true_spans = fuse_adjacent_tag(true_spans)
    # pred_spans, reconstructed_text, count = gather_tagged_entities(
    #     abstract, llm_annotation, tag2label_st21pv
    # )
    # pred_spans = fuse_adjacent_tag(pred_spans)  
    # pred_spans = sorted(noisy_data[pmid]['pred_spans'], key=lambda x: x['start'])
    pred_spans = [p for p in pred_spans if p["start"] != -1]
    pred_spans = sorted(pred_spans, key=lambda x: x['start'])
    f1, precision, recall, correct_count, noise_count, results, noise_types = compute_noise(pred_spans, true_spans)
    noise = 1-f1
    # if reconstructed_text != abstract: # 1) if reconstructed text is not the same as the abstract, skip because the corruption really likely went wrong.
    #     nb_invalids_hallucination.add(pmid)
    #     continue
    if len(pred_spans) == 0: # 2) if pred_spans have less than 2 annotations, skip because no minimum of annotations per abstract in the dataset is 4.
        nb_invalids_empty.add(pmid)
        continue
    total_correct_count += correct_count
    total_noise_count += noise_count # Add noise only if it's not among the previous errors 
    total_n_true += len(true_spans)
    final_noise[pmid] = noise
    for span in results:
        if span['correct'] == False:
            error_analysis[span['error_type']] += 1
        else :
            nb_corrects += 1
    full_results.append({"noise" : noise,
                         "f1" :f1, 
                         "precision" : precision,
                         "recall" : recall,
                         "correct_count" : correct_count,
                         "noise_count" : noise_count, 
                         "tagged_text" : llm_annotation,
                         "pred_spans" : pred_spans,
                         "true_spans" : true_spans,
                         "results" : results,
                         "noise_types" : noise_types,
                         "pmid" : pmid})
    
avg_noise = np.mean(list(final_noise.values())) * len(final_noise) / len(data_train_v1)
final_precision = total_correct_count / (total_correct_count + total_noise_count)
final_recall = total_correct_count / total_n_true
final_f1 = 2 * final_precision * final_recall / (final_precision + final_recall)
normalized_noise = 1-final_f1
print(f"final_f1 : {final_f1} | final_precision : {final_precision} | final_recall : {final_recall}")
print("Noise (each dataset contributed equally):", avg_noise) # Noise over the whole dataset
print("Normalized Noise (Dataset with more tags contributes more):", normalized_noise) # Noise over the whole dataset
print(f"Number of invalid corrupted abstracts : \n len(pred_spans)==0 : {len(nb_invalids_empty)} || Reconstructed != original : {len(nb_invalids_hallucination)} || Untouched abstract : {len(nb_untouched)} || Formatting issue : {len(raw_noisy_data)-len(noisy_data)} \n")
print("Error per type :", dict(sorted(error_analysis.items(), key = lambda x : x[1], reverse=True)))
print("Number of corrects :", nb_corrects)
print("Total noise count :", total_noise_count, "|| Total noise count if we add 'missing' errors :", sum(error_analysis.values()))
print("Total number of true spans :", total_n_true)

### Sanity check : compute_noise and seqeval gets the same result

In [ ]:
pmids = [el["pmid"] for el in llm_output]
pmid2pred_spans = {}

for pmid in noisy_data:
    pred_spans = noisy_data[pmid]['pred_spans']
    pred_spans = [p for p in pred_spans if p["start"] != -1]
    pred_spans = sorted(pred_spans, key=lambda x: x['start'])
    pmid2pred_spans[pmid] = pred_spans

results, df, nb_valid_eval, empty = compute_metrics(
    pmids=pmids,
    data=data,
    tag_to_label=tag2label_st21pv,
    pmid2pred_spans=pmid2pred_spans,
    debug=False,
)

print("Number of valid evaluations: ", nb_valid_eval)
print("Results : ", results)

### Check Errors per type for one abstract

In [ ]:
idx = 15
full_results[idx]["noise_types"]
pmid = full_results[idx]["pmid"]

abstract = data[pmid]['input']

from IPython.display import HTML, display
display(HTML(f"<div style='white-space: pre-wrap; word-wrap: break-word;'>{abstract}</div>"))



In [ ]:
output = data[pmid]['output']
display(HTML(f"<div style='white-space: pre-wrap; word-wrap: break-word;'>{output}</div>"))

In [ ]:
true_spans = data_train_v1[pmid]['spans']


# 2nd load

In [ ]:
# model_version = "phi-4_v16"
# model_version = "phi-4_v15_noise=0.7"
# model_version = "gpt-4o-2024-11-20_noise=0.2"
# model_version = "BioLinkBERT-base_v1_noise=0.4"
# model_version = "BioLinkBERT-base_v1_noise=0.1_v2"
# model_version = "BioLinkBERT-base_train_v1"
# model_version = "BioLinkBERT-base_train_v1_noise=0.5"
# model_version = "Mistral-Nemo-Instruct-2407_v5"
# model_version = "Llama-3.1-8B-Instruct_v2"
# model_version = "gpt-4o-2024-11-20_dynamic=True_k=10_v11_noise=0.6"
model_version = "gemma-3-12b-it_v2"
# model_version = "gemma-3-12b-it_v2_noise=0.8"
# model_version = "gemma-3-12b-it_v2_noise=0.1496_dist=wl"
# model_version = "Mistral-Small-3.1-24B-Instruct-2503_dynamic=True_k=5_v3"
noisy_data_path = f"../data/{dataset_name}/{dataset_name}_{model_version}.json"
with open(noisy_data_path, "r") as f:
    noisy_data = json.load(f)
print("Number of noisy data :", len(noisy_data))

noisy_data = {pmid: noisy_data[pmid] for pmid in noisy_data if noisy_data[pmid]['split']=='train'} 
data_train_v1 = {pmid: data[pmid] for pmid in data if data[pmid]["split"] == "train"}

total_n_true = 0
for pmid in noisy_data:
    true_spans = data_train_v1[pmid]['spans']
    # true_spans = fuse_adjacent_tag(true_spans)
    total_n_true += len(true_spans)

In [ ]:
pmids2check = []

for idx, pmid in enumerate(noisy_data):
    llm_annotation = noisy_data[pmid]["output"]
    abstract, true_spans = data[pmid]['input'], data_train_v1[pmid]['spans']
    true_spans = fuse_adjacent_tag(true_spans)
    real_annotation = create_tagged_text(abstract, true_spans)
    pred_spans, reconstructed_text, count = gather_tagged_entities(
        abstract, llm_annotation, tag2label_st21pv
    ) 
    # pred_spans = [p for p in pred_spans if p["start"] != -1]
    pred_spans = [p for p in pred_spans if (p["start"] != -1 and (p["end"]-p["start"]<70))]
    pred_spans = sorted(pred_spans, key=lambda x: x['start'])
    pred_spans = fuse_adjacent_tag(pred_spans)
    f1, precision, recall, correct_count, noise_count, results, noise_types = compute_noise(pred_spans, true_spans)
    
    y_true_seqeval = label_tokens_from_offsets(text = abstract, annotations = true_spans)
    y_pred_seqeval = label_tokens_from_offsets(text = abstract, annotations = pred_spans)
    results = seqeval.compute(predictions=[y_pred_seqeval], references=[y_true_seqeval])
    precision_2 = results["overall_precision"]
    recall_2 = results["overall_recall"]
    f1_2 = results["overall_f1"]
    accuracy_2 = results["overall_accuracy"]
    
    if abs(f1-f1_2) > 0.05:
    # if f1 != f1_2:
        pmids2check.append((idx, pmid))


# f1, precision, recall, correct_count, noise_count, len(true_spans), results


In [ ]:
pmid = pmids2check[1][1]
# pmid = "27620334"
print(pmid)
llm_annotation = noisy_data[pmid]["output"]
abstract, true_spans = data[pmid]['input'], data_train_v1[pmid]['spans']
true_spans = fuse_adjacent_tag(true_spans)

real_annotation = create_tagged_text(abstract, true_spans)
pred_spans, reconstructed_text, count = gather_tagged_entities(
    abstract, llm_annotation, tag2label_st21pv
)
# pred_spans = [p for p in pred_spans if p["start"] != -1]
pred_spans = [p for p in pred_spans if (p["start"] != -1 and (p["end"]-p["start"]<70))]
pred_spans = sorted(pred_spans, key=lambda x: x['start'])
pred_spans = fuse_adjacent_tag(pred_spans)
reconstructed_annotation = create_tagged_text(abstract, pred_spans)
f1, precision, recall, correct_count, noise_count, results, noise_types = compute_noise(pred_spans, true_spans)

y_true_seqeval = label_tokens_from_offsets(text = abstract, annotations = true_spans)
y_pred_seqeval = label_tokens_from_offsets(text = abstract, annotations = pred_spans)
results2 = seqeval.compute(predictions=[y_pred_seqeval], references=[y_true_seqeval])
precision_2, recall_2, f1_2, accuracy_2 = results2["overall_precision"], results2["overall_recall"], results2["overall_f1"], results2["overall_accuracy"]

print("f1 :", f1, "|| f1_2 :", f1_2)
print("precision :", precision, "|| precision_2 :", precision_2)
print("recall :", recall, "|| recall_2 :", recall_2)
print("correct_count :", correct_count, "noise_count :", noise_count, "|| len(true_spans) :", len(true_spans))



In [ ]:
recall_2 * len(true_spans), 56/(56+28)

In [ ]:
reduction = 1
new_spans = reduce_noise_per_abstract(correct_count, 
                                    noise_count, 
                                    pred_spans, 
                                    true_spans, 
                                    reduction, 
                                    results, 
                                #  debug=True
                                    )


In [ ]:
f1_v2, precision_v2, recall_v2, correct_count_v2, noise_count_v2, results_v2, noise_types_v2 = compute_noise(new_spans, true_spans, debug=False)
noise_v2 = 1-f1_v2
print(f"\nAfter reduction \n Noise: {noise_v2} | Number of correct count: {correct_count_v2} | Number of noise count: {noise_count_v2} | Number of true: {len(true_spans)} \n f1: {f1_v2} | precision:, {precision_v2} | recall: {recall_v2}")

In [ ]:
new_spans

In [ ]:
results_v2

In [ ]:
reduction = 1
new_spans_v2 = reduce_noise_per_abstract(correct_count_v2, 
                                    noise_count_v2, 
                                    new_spans, 
                                    true_spans, 
                                    reduction, 
                                    results_v2, 
                                #  debug=True
                                    )


In [ ]:
f1_v3, precision_v3, recall_v3, correct_count_v3, noise_count_v3, results_v3, noise_types_v3 = compute_noise(new_spans_v2, true_spans, debug=False)
noise_v3 = 1-f1_v3
print(f"\nAfter reduction \n Noise: {noise_v3} | Number of correct count: {correct_count_v3} | Number of noise count: {noise_count_v3} | Number of true: {len(true_spans)} \n f1: {f1_v3} | precision:, {precision_v3} | recall: {recall_v3}")

In [ ]:
results_v3

In [ ]:
true_spans

In [ ]:
if new_spans == true_spans:
    print("True spans are the same")

In [ ]:
errors_to_remove =  noise_types

reduction = 1
new_spans2 = reduce_noise_per_abstract_distribution(
  correct_count, 
  noise_count, 
  pred_spans, 
  true_spans, 
  reduction, 
  results, 
  noise_types, 
  errors_to_remove)
f1_v2, precision_v2, recall_v2, correct_count_v2, noise_count_v2, results_v2, noise_types_v2 = compute_noise(new_spans2, true_spans)
noise_v2 = 1 - f1_v2

print(f"\nBefore reduction \n Noise: {noise} | Number of correct count: {correct_count} | Number of noise count: {noise_count} | Number of true: {len(true_spans)} \n f1: {f1} | precision:, {precision} | recall: {recall}" )
print(" Error distribution before reduction :", noise_types)
print(f"\nAfter reduction \n Noise: {noise_v2} | Number of correct count: {correct_count_v2} | Number of noise count: {noise_count_v2} | Number of true: {len(true_spans)} \n f1: {f1_v2} | precision:, {precision_v2} | recall: {recall_v2}")
print(" Error distribution after reduction :", noise_types_v2)



In [ ]:
if new_spans2 == true_spans:
    print("True spans are the same")

## Replace invalid (empty pred_spans)

In [ ]:
model_version2 = "Mistral-Small-3.1-24B-Instruct-2503_dynamic=True_k=5_v3"
noisy_data_path2 = f"../data/{dataset_name}/{dataset_name}_{model_version2}.json"
with open(noisy_data_path2, "r") as f:
    noisy_data2 = json.load(f)
print("Number of noisy data :", len(noisy_data2))

In [ ]:
print(len(noisy_data))
noisy_data_new = {}
for pmid in noisy_data:
    pred_spans = noisy_data[pmid]['spans']
    pred_spans = [p for p in pred_spans if (p["start"] != -1 and (p["end"]-p["start"]<70))]
    if len(pred_spans) > 2 :
        noisy_data_new[pmid] = noisy_data[pmid]
print(len(noisy_data_new))

In [ ]:
for pmid in noisy_data2:
    if pmid not in noisy_data_new:
        noisy_data_new[pmid] = noisy_data2[pmid]

In [ ]:
noisy_data = noisy_data_new
noisy_data = {pmid: noisy_data[pmid] for pmid in noisy_data if noisy_data[pmid]['split']=='train'} 
print(len(noisy_data))

## 2nd load next

In [ ]:
final_noise = {}
nb_invalids_noise = set()
nb_invalids_empty = set()
nb_invalids_hallucination = set()
nb_untouched = set()
nb_corrects = 0
total_noise_count = 0
total_correct_count = 0
# total_n_true = 0

full_results = []

error_analysis = defaultdict(int)

# for pmid in list(noisy_data.keys())[1:2]:
for pmid in noisy_data:
    llm_annotation = noisy_data[pmid]["output"]
    abstract, true_spans = data[pmid]['input'], data_train_v1[pmid]['spans']
    # # true_spans = fuse_adjacent_tag(true_spans)
    pred_spans, reconstructed_text, count = gather_tagged_entities(
        abstract, llm_annotation, tag2label_st21pv
    ) 
    # pred_spans = noisy_data[pmid]['spans']
    # pred_spans = [p for p in pred_spans if p["start"] != -1]
    pred_spans = [p for p in pred_spans if (p["start"] != -1 and (p["end"]-p["start"]<70))]
    pred_spans = sorted(pred_spans, key=lambda x: x['start'])
    # pred_spans = fuse_adjacent_tag(pred_spans)
    # pred_spans = sorted(noisy_data[pmid]['spans'], key=lambda x: x['start'])
    f1, precision, recall, correct_count, noise_count, results, noise_types = compute_noise(pred_spans, true_spans)
    noise = 1-f1
    # if reconstructed_text != abstract: # 1) if reconstructed text is not the same as the abstract, skip because the corruption really likely went wrong.
    #     nb_invalids_hallucination.add(pmid)
    #     continue
    if len(pred_spans) == 0: # 2) if pred_spans have less than 2 annotations, skip because no minimum of annotations per abstract in the dataset is 4.
        nb_invalids_empty.add(pmid)
        # print(pmid)
        continue
    total_correct_count += correct_count
    total_noise_count += noise_count # Add noise only if it's not among the previous errors 
    # total_n_true += len(true_spans)
    final_noise[pmid] = noise
    for span in results:
        if span['correct'] == False:
            error_analysis[span['error_type']] += 1
        else :
            nb_corrects += 1
    full_results.append({"noise" : noise,
                         "f1" :f1, 
                         "precision" : precision,
                         "recall" : recall,
                         "correct_count" : correct_count,
                         "noise_count" : noise_count, 
                         "tagged_text" : llm_annotation,
                         "pred_spans" : pred_spans,
                         "true_spans" : true_spans,
                         "results" : results,
                         "noise_types" : noise_types,
                         "pmid" : pmid})

avg_noise = np.mean(list(final_noise.values())) * len(final_noise) / len(data_train_v1)
final_precision = total_correct_count / (total_correct_count + total_noise_count)
final_recall = total_correct_count / total_n_true
final_f1 = 2 * final_precision * final_recall / (final_precision + final_recall)
normalized_noise = 1-final_f1
print(f"Final F1 : {final_f1} | Final Precision : {final_precision} | Final Recall : {final_recall}" )
print("Noise (each dataset contributed equally):", avg_noise) # Noise over the whole dataset
print("Normalized Noise (Dataset with more tags contributes more):", normalized_noise) # Noise over the whole dataset
print(f"Number of invalid corrupted abstracts : \n len(pred_spans) < 3 : {len(nb_invalids_empty)} || Reconstructed != original : {len(nb_invalids_hallucination)} || Untouched abstract : {len(nb_untouched)}") # || Formatting issue : {len(raw_noisy_data)-len(noisy_data)} \n")
print("Error per type :", dict(sorted(error_analysis.items(), key = lambda x : x[1], reverse=True)))
print("Number of corrects :", nb_corrects)
print("Total noise count :", total_noise_count, "|| Total noise count if we add 'missing' errors :", sum(error_analysis.values())) # Total noise count including all errors
print("Total number of true spans :", total_n_true)

In [ ]:
nb_invalids_empty = list(nb_invalids_empty)

In [ ]:
llm_annotation = noisy_data[pmid]["output"]
from IPython.display import HTML, display
display(HTML(f"<div style='white-space: pre-wrap; word-wrap: break-word;'>{llm_annotation}</div>"))

In [ ]:
pmid = nb_invalids_empty[0]
noisy_data[pmid]['spans']

In [ ]:
abstract, true_spans = data[pmid]['input'], data_train_v1[pmid]['spans']
abstract

In [ ]:
data_check = {pmid : item for pmid, item in noisy_data.items() if len(item['spans']) < 3}
data_check

In [ ]:
noisy_data

In [ ]:
# THIS IS BIASED AS A LOT OF ERRORS ARE COUNTED TWICE (ESPECIALLY MULTIPLE ENTITIES)
# expected correct = current correct + missing corrected + multiple entities corrected x 2 + wrong labels corrected + wrong overlap corrected
expected_correct = nb_corrects + error_analysis['missing'] + error_analysis['multiple_entities'] * 2 + error_analysis['wrong_label'] + error_analysis['wrong_overlap']
print("Expected correct :", expected_correct)
# expected noise = current noise - missing corrected - multiple entities corrected - wrong labels corrected - wrong overlap corrected - invalid corrected
expected_noise = total_noise_count - error_analysis['multiple_entities'] - error_analysis['wrong_label'] - error_analysis['wrong_overlap'] - error_analysis['invalid_tag']
print("Expected noise :", expected_noise)

In [ ]:
noisy_data

## Sanity Check

In [ ]:
pmids = [pmid for pmid in noisy_data]
pmid2pred_spans = {pmid: val["spans"] for pmid, val in noisy_data.items()}

results, df, nb_valid_eval = compute_metrics(
    pmids=pmids,
    data=data,
    tag_to_label=tag2label_st21pv,
    pmid2pred_spans=pmid2pred_spans,
    debug=False,
)

print("Number of valid evaluations: ", nb_valid_eval)
print("Results : ", results)

# Add more noise by taking results from different corruptions

### Noise reduction on 1 abstract

In [ ]:
idx = 500
abstract, annotated_abstract, true_spans = data_train[idx]["input"], data_train[idx]["output"], data_train[idx]['spans']
print("pmid : ", data_train[idx]['pmid'])
pmid = data_train[idx]['pmid']
# pred_spans = noisy_data[pmid]['pred_spans']
pred_spans = noisy_data[pmid]['spans']
pred_spans = sorted(pred_spans, key=lambda x: x["start"])
f1, precision, recall, correct_count, noise_count, results, noise_types = compute_noise(pred_spans, true_spans, debug=False)
noise = 1-f1
print(f"\nBefore reduction \n Noise: {noise} | Number of correct count: {correct_count} | Number of noise count: {noise_count} | Number of true: {len(true_spans)} \n f1: {f1} | precision:, {precision} | recall: {recall}" )


In [ ]:
reduction = 0.5
new_spans = reduce_noise_per_abstract(correct_count, 
                                             noise_count, 
                                             pred_spans, 
                                             true_spans, 
                                             reduction, 
                                             results, 
                                            #  debug=True
                                             )


In [ ]:
f1_v2, precision_v2, recall_v2, correct_count_v2, noise_count_v2, results_v2, noise_types_v2 = compute_noise(new_spans, true_spans, debug=False)
noise_v2 = 1-f1_v2
print(f"\nAfter reduction \n Noise: {noise_v2} | Number of correct count: {correct_count_v2} | Number of noise count: {noise_count_v2} | Number of true: {len(true_spans)} \n f1: {f1_v2} | precision:, {precision_v2} | recall: {recall_v2}")

### Noise reduction on whole dataset

In [ ]:
distribution = "N"
version = "v1"
reduction = 0.832
wanted_noise = 0.1

new_full_results = reduce_noise(
    full_results=full_results, 
    wanted_noise = wanted_noise, 
    reduction = reduction, 
    total_n_true = total_n_true,
    debug=False)

In [ ]:
new_final_noise = {}
new_nb_invalids = []
new_total_noise_count = 0
new_total_correct_count = 0
new_nb_corrects = 0

new_error_analysis = defaultdict(int)

for res in new_full_results:
    pmid = res['pmid']
    new_final_noise[pmid] = res['noise']
    new_total_noise_count += res['noise_count']
    new_total_correct_count += res['correct_count'] 
    for span in res['results']:
        if span['correct'] == False:
            new_error_analysis[span['error_type']] += 1
        else:
            new_nb_corrects += 1

assert new_nb_corrects == new_total_correct_count, f"{new_nb_corrects} != {new_total_correct_count}"        
            
new_final_precision = new_total_correct_count / (new_total_correct_count + new_total_noise_count)
new_final_recall = new_total_correct_count / total_n_true
new_final_f1 = 2 * new_final_precision * new_final_recall / (new_final_precision + new_final_recall)
new_normalized_noise = 1-new_final_f1

new_avg_noise = np.mean(list(new_final_noise.values())) * len(final_noise) / len(data_train_v1)
print("Noise (each dataset contributed equally):", new_avg_noise) # Noise over the whole dataset
print("Normalized Noise (Dataset with more tags contributes more):", new_normalized_noise)
print(f"Number of invalid corrupted abstracts : \n len(pred_spans) < 3 : {len(nb_invalids_empty)} || Reconstructed != original : {len(nb_invalids_hallucination)} || Untouched abstract : {len(nb_untouched)}") # || Formatting issue : {len(raw_noisy_data)-len(noisy_data)}")
print("Error per type :", dict(sorted(new_error_analysis.items(), key = lambda x : x[1], reverse=True)))
print("Number of corrects :", new_nb_corrects)
print("Total noise count :", new_total_noise_count, "|| Total noise count if we add 'missing' errors :", sum(new_error_analysis.values()))
print("Total number of true spans :", total_n_true)


# Reduce_noise_per_abstract_distribution and Reduce_noise_distribution

### Noise reduction on one abstract (Distribution)

In [ ]:
pmids2check = []
pmids2check2 = []
for idx in range(len(data_train)):
    abstract, annotated_abstract, true_spans = data_train[idx]["input"], data_train[idx]["output"], data_train[idx]['spans']
    # true_spans = fuse_adjacent_tag(true_spans)
    # print("pmid : ", data_train[idx]['pmid'])
    pmid = data_train[idx]['pmid']
    # pred_spans = noisy_data[pmid]['pred_spans']
    pred_spans = noisy_data[pmid]['spans']
    pred_spans = [p for p in pred_spans if (p["start"] != -1 and (p["end"]-p["start"]<70))]
    pred_spans = sorted(pred_spans, key=lambda x: x['start'])
    # pred_spans = fuse_adjacent_tag(pred_spans)
    f1, precision, recall, correct_count, noise_count, results, noise_types = compute_noise(pred_spans, true_spans)
    noise = 1 - f1

    # errors_to_remove =  {'wrong_label': 0,
    #   'wrong_overlap': 1,
    #   'multiple_entities': 1,
    #   'invalid_tag': 1,
    #   'missing': 40}

    errors_to_remove =  noise_types

    reduction = 1
    new_spans = reduce_noise_per_abstract_distribution(
      correct_count, 
      noise_count, 
      pred_spans, 
      true_spans, 
      reduction, 
      results, 
      noise_types, 
      errors_to_remove)
    f1_v2, precision_v2, recall_v2, correct_count_v2, noise_count_v2, results_v2, noise_types_v2 = compute_noise(new_spans, true_spans)
    noise_v2 = 1 - f1_v2

    if noise_v2 != 0 :
        pmids2check.append((idx, pmid))
      
    if correct_count_v2 > len(true_spans):
        pmids2check2.append((idx, pmid))



In [ ]:
pmids2check = []
for idx in range(len(data_train)):
    abstract, annotated_abstract, true_spans = data_train[idx]["input"], data_train[idx]["output"], data_train[idx]['spans']
    # true_spans = fuse_adjacent_tag(true_spans)
    # print("pmid : ", data_train[idx]['pmid'])
    pmid = data_train[idx]['pmid']
    # pred_spans = noisy_data[pmid]['pred_spans']
    pred_spans = noisy_data[pmid]['spans']
    pred_spans = [p for p in pred_spans if (p["start"] != -1 and (p["end"]-p["start"]<70))]
    pred_spans = sorted(pred_spans, key=lambda x: x['start'])
    # pred_spans = fuse_adjacent_tag(pred_spans)
    f1, precision, recall, correct_count, noise_count, results, noise_types = compute_noise(pred_spans, true_spans)
    noise = 1 - f1

    # errors_to_remove =  {'wrong_label': 0,
    #   'wrong_overlap': 1,
    #   'multiple_entities': 1,
    #   'invalid_tag': 1,
    #   'missing': 40}

    errors_to_remove =  noise_types

    reduction = 0.5
    
    new_spans_v1 = reduce_noise_per_abstract_distribution(
      correct_count, 
      noise_count, 
      pred_spans, 
      true_spans, 
      reduction, 
      results, 
      noise_types, 
      errors_to_remove)
    f1_v1, precision_v1, recall_v1, correct_count_v1, noise_count_v1, results_v1, noise_types_v1 = compute_noise(new_spans_v1, true_spans)
    noise_v1 = 1 - f1_v1
    
    new_spans_v2 = reduce_noise_per_abstract_distribution2(
      correct_count, 
      noise_count, 
      pred_spans, 
      true_spans, 
      reduction, 
      results, 
      noise_types, 
      errors_to_remove)
    f1_v2, precision_v2, recall_v2, correct_count_v2, noise_count_v2, results_v2, noise_types_v2 = compute_noise(new_spans_v2, true_spans)
    noise_v2 = 1 - f1_v2

    if noise_v1 != noise_v2 :
        pmids2check.append((idx, pmid))
      



In [ ]:
len(pmids2check)

In [ ]:
len(pmids2check), len(pmids2check2), pmids2check, pmids2check2

In [ ]:
idx = 177

abstract, annotated_abstract, true_spans = data_train[idx]["input"], data_train[idx]["output"], data_train[idx]['spans']
# true_spans = fuse_adjacent_tag(true_spans)
pmid = data_train[idx]['pmid']
pred_spans = noisy_data[pmid]['spans']
pred_spans = [p for p in pred_spans if (p["start"] != -1 and (p["end"]-p["start"]<70))]
pred_spans = sorted(pred_spans, key=lambda x: x['start'])
# pred_spans = fuse_adjacent_tag(pred_spans)
f1, precision, recall, correct_count, noise_count, results, noise_types = compute_noise(pred_spans, true_spans)
noise = 1 - f1

errors_to_remove =  noise_types

reduction = 1
new_spans = reduce_noise_per_abstract_distribution(
    correct_count, 
    noise_count, 
    pred_spans, 
    true_spans, 
    reduction, 
    results, 
    noise_types, 
    errors_to_remove)
f1_v2, precision_v2, recall_v2, correct_count_v2, noise_count_v2, results_v2, noise_types_v2 = compute_noise(new_spans, true_spans)
noise_v2 = 1 - f1_v2


print("idx :", idx, "PMID :", pmid)
print(f"\nBefore reduction \n Noise: {noise} | Number of correct count: {correct_count} | Number of noise count: {noise_count} | Number of true: {len(true_spans)} \n f1: {f1} | precision:, {precision} | recall: {recall}" )
print(" Error distribution before reduction :", noise_types)
print(f"\nAfter reduction \n Noise: {noise_v2} | Number of correct count: {correct_count_v2} | Number of noise count: {noise_count_v2} | Number of true: {len(true_spans)} \n f1: {f1_v2} | precision:, {precision_v2} | recall: {recall_v2}")
print(" Error distribution after reduction :", noise_types_v2)




In [ ]:
for d1 in true_spans:
    if d1 not in new_spans:
        print(d1)


In [ ]:
pred_spans

In [ ]:
# new_spans

In [ ]:
results

In [ ]:
results_v2

In [ ]:
true_spans

In [ ]:
# P = 53/69
# R = 53/65
# F1 = 2 * P * R / (P + R) if (P + R) > 0 else 0
# N = 1 - F1
# N

### Noise reduction on whole dataset (distribution)

In [ ]:
reduction = 0.75
wanted_noise = 0.1
# distribution = "wl"
distribution = "m"
key2dist = {"wl" : "wrong_label", "m" : "missing", "wo" : "wrong_overlap", "me" : "multiple_entities", "it" : "invalid_tag"}
version = "v1" # Without correction to reduce_noise_per_abstract function.

target_distribution = all_distributions[distribution]

initial_correct_counts = nb_corrects
initial_noise_counts = total_noise_count
n_true_spans = total_n_true

A_mid, final_noisee, final_correct, final_noise_count, final_target_numbers = compute_noise_coefficient(
    wanted_noise,
    error_analysis,
    initial_correct_counts,
    initial_noise_counts,
    n_true_spans,
    target_distribution,
    # debug=True
)

mini = 8515/2
proportions = {
    'missing': 0.5,
    'wrong_overlap': 0.125,
    'invalid_tag': 0.125,
    'wrong_label': 0.125,
    'multiple_entities': 0.125
}

final_target_numbers = {
    key : int(mini * val/proportions["multiple_entities"]) for key, val in proportions.items()
}

# final_target_numbers = {
#     key : int(error_analysis[key2dist[distribution]]/4) if key != key2dist[distribution] else error_analysis[key] for key in error_analysis
# }
final_target_numbers = {key : int(val * 1.105 /2.13) for key, val in final_target_numbers.items()}
final_target_numbers

In [ ]:
print(f"initial noise : {normalized_noise} | initial_correct : {initial_correct_counts} | initial_noise_count : {initial_noise_counts}")
ordered_error_start = {key : error_analysis[key] for key in target_distribution}
print(f"initial error counts : {ordered_error_start}")

print(f"coefficient : {A_mid}")
print(f"Expected final noise : {final_noisee} | Expected final_correct : {final_correct} | Expected final_noise_count : {final_noise_count} | total_n_true : {n_true_spans}")
print(f"Expected final target numbers : {final_target_numbers}")
added_correct = ((error_analysis["multiple_entities"] - final_target_numbers["multiple_entities"]) * 2 +
                         (error_analysis["wrong_label"] - final_target_numbers["wrong_label"]) +
                         (error_analysis["wrong_overlap"] - final_target_numbers["wrong_overlap"]) +
                         (error_analysis["missing"] - final_target_numbers["missing"]))

# Noise reduction: remove -1 for each correction for "multiple_entities", "wrong_label", "wrong_overlap", "invalid_tag"
reduced_noise = ((error_analysis["multiple_entities"] - final_target_numbers["multiple_entities"]) +
                    (error_analysis["wrong_label"] - final_target_numbers["wrong_label"]) +
                    (error_analysis["wrong_overlap"] - final_target_numbers["wrong_overlap"]) +
                    (error_analysis["invalid_tag"] - final_target_numbers["invalid_tag"]))
real_correct_count = initial_correct_counts + added_correct
real_noise_count = initial_noise_counts - reduced_noise 
print(f"Expected real correct count :", real_correct_count)
print(f"Expected real noise count :", real_noise_count)

# assert final_noise_count == real_noise_count, f"{final_noise_count} != {real_noise_count}. Final noise count should be equal to the sum of final target numbers minus missing."
# assert final_correct == real_correct_count, f"{final_correct} != {real_correct_count}. Final correct count should be equal to the sum of the initial error analysis minus the final target numbers."
# assert all(
#     final_target_numbers[key] <= error_analysis[key]
#     for key in final_target_numbers
# ), f"At least one final target number is greater than the initial count. Details: {final_target_numbers}, Initial: {error_analysis}"


In [ ]:
initial_correct_counts, added_correct

In [ ]:
correct = final_correct
wrong = final_noise_count
P = correct / (correct + wrong) if (correct + wrong) > 0 else 0
R = correct / total_n_true if total_n_true > 0 else 0
F = (2 * P * R / (P + R)) if (P + R) > 0 else 0
N = 1 - F
print(f"P: {P:.3f}, R: {R:.3f}, F: {F:.3f}, N: {N:.3f}")


In [ ]:
# Correct = total possible - wl - wo - 2 * me - missing
corrrrect = (
    n_true_spans -
    (final_target_numbers["wrong_label"] +
    final_target_numbers["wrong_overlap"] +
    final_target_numbers["multiple_entities"] * 2)
)
print(f"Total correct : {corrrrect}")
# Wrong = wl + wo + me + it
wrrrrong = (
    final_target_numbers["wrong_label"] +
    final_target_numbers["wrong_overlap"] +
    final_target_numbers["multiple_entities"] +
    final_target_numbers["invalid_tag"]
)
print(f"Total wrong : {wrrrrong}")

In [ ]:
new_full_results = reduce_noise_distribution(
    full_results=full_results, 
    wanted_noise = wanted_noise, 
    reduction = reduction, 
    total_n_true = total_n_true,
    total_noise_types=error_analysis,
    target_numbers=final_target_numbers,
    # debug=True,
    # debug=False
    )



In [ ]:
new_final_noise = {}
new_nb_invalids = []
new_total_noise_count = 0
new_total_correct_count = 0
new_nb_corrects = 0

new_error_analysis = defaultdict(int)

for res in new_full_results:
    pmid = res['pmid']
    new_final_noise[pmid] = res['noise']
    new_total_noise_count += res['noise_count']
    new_total_correct_count += res['correct_count'] 
    for span in res['results']:
        if span['correct'] == False:
            new_error_analysis[span['error_type']] += 1
        else:
            new_nb_corrects += 1



In [ ]:
assert new_nb_corrects == new_total_correct_count, f"{new_nb_corrects} != {new_total_correct_count}"        
            
new_final_precision = new_total_correct_count / (new_total_correct_count + new_total_noise_count)
new_final_recall = new_total_correct_count / total_n_true
new_final_f1 = 2 * new_final_precision * new_final_recall / (new_final_precision + new_final_recall)
new_normalized_noise = 1-new_final_f1

new_avg_noise = np.mean(list(new_final_noise.values())) * len(new_final_noise) / len(data_train_v1)
print(f"Final F1 : {new_final_f1} | Final Precision : {new_final_precision} | Final Recall : {new_final_recall}" )
print("Noise (each dataset contributed equally):", new_avg_noise) # Noise over the whole dataset
print("Normalized Noise (Dataset with more tags contributes more):", new_normalized_noise)
print(f"Number of invalid corrupted abstracts : \n len(pred_spans) < 3 : {len(nb_invalids_empty)} || Reconstructed != original : {len(nb_invalids_hallucination)} || Untouched abstract : {len(nb_untouched)}")   # || Formatting issue : {len(raw_noisy_data)-len(noisy_data)} \n")
print("Error per type :", dict(sorted(new_error_analysis.items(), key = lambda x : x[1], reverse=True)))
print(f"Number of corrects : {new_nb_corrects} || Target number of corrects : {final_correct}")
print(f"Total noise count : {new_total_noise_count} || Target noise counts : {final_noise_count} || Total noise count if we add 'missing' errors : {sum(new_error_analysis.values())}")
print("Total number of true spans :", total_n_true)

In [ ]:
ordered_error_start = {key : error_analysis[key] for key in target_distribution}
print("Starting errors : \n", ordered_error_start) 

print("target_numbers : \n", final_target_numbers)

ordered_error_end = {key : new_error_analysis[key] for key in target_distribution}
print("Current errors: \n", ordered_error_end)

In [ ]:
end_distribution = {key:new_error_analysis[key]/np.sum(list(new_error_analysis.values())) for key in target_distribution}
end_distribution

In [ ]:
# THIS IS BIASED AS A LOT OF ERRORS ARE COUNTED TWICE (ESPECIALLY MULTIPLE ENTITIES)
# expected correct = current correct + missing corrected + multiple entities corrected x 2 + wrong labels corrected + wrong overlap corrected
new_expected_correct = new_nb_corrects + new_error_analysis['missing'] + new_error_analysis['multiple_entities'] * 2 + new_error_analysis['wrong_label'] + new_error_analysis['wrong_overlap']
print("Expected correct :", new_expected_correct)
# expected noise = current noise - missing corrected - multiple entities corrected - wrong labels corrected - wrong overlap corrected - invalid corrected
new_expected_noise = new_total_noise_count - new_error_analysis['multiple_entities'] - new_error_analysis['wrong_label'] - new_error_analysis['wrong_overlap'] - new_error_analysis['invalid_tag']
print("Expected noise :", new_expected_noise)

## Second Pass

In [ ]:
final_target_numbers2 = {
    key : final_target_numbers[key] if final_target_numbers[key] < ordered_error_end[key] else ordered_error_end[key] # Normalize the target distribution to sum to 1
    for key in final_target_numbers
}

In [ ]:
print(f"Current state of errors : \n {ordered_error_end}") 
print(f"Final target numbers : \n {final_target_numbers}")
print(f"Final target numbers adjusted to make sure reduce_noise_distribution run without error : \n {final_target_numbers2}")

In [ ]:
reduction2 = 0.75
new_full_results2 = reduce_noise_distribution(
    full_results=new_full_results, 
    wanted_noise = wanted_noise, 
    reduction = reduction2, 
    total_n_true = total_n_true,
    total_noise_types=new_error_analysis,
    target_numbers=final_target_numbers2,
    # debug=True,
    )



In [ ]:
new_final_noise2 = {}
new_nb_invalids2 = []
new_total_noise_count2 = 0
new_total_correct_count2 = 0
new_nb_corrects2 = 0

new_error_analysis2 = defaultdict(int)

for res in new_full_results2:
    pmid = res['pmid']
    new_final_noise2[pmid] = res['noise']
    new_total_noise_count2 += res['noise_count']
    new_total_correct_count2 += res['correct_count'] 
    for span in res['results']:
        if span['correct'] == False:
            new_error_analysis2[span['error_type']] += 1
        else:
            new_nb_corrects2 += 1



In [ ]:
assert new_nb_corrects2 == new_total_correct_count2, f"{new_nb_corrects2} != {new_total_correct_count2}"        
            
new_final_precision2 = new_total_correct_count2 / (new_total_correct_count2 + new_total_noise_count2)
new_final_recall2 = new_total_correct_count2 / total_n_true
new_final_f1_2 = 2 * new_final_precision2 * new_final_recall2 / (new_final_precision2 + new_final_recall2)
new_normalized_noise2 = 1-new_final_f1_2

new_avg_noise2 = np.mean(list(new_final_noise2.values())) * len(new_final_noise2) / len(data_train_v1)
print(f"Final F1 : {new_final_f1_2} | Final Precision : {new_final_precision2} | Final Recall : {new_final_recall2}" )
print("Noise (each dataset contributed equally):", new_avg_noise2) # Noise over the whole dataset
print("Normalized Noise (Dataset with more tags contributes more):", new_normalized_noise2)
print("Error per type :", dict(sorted(new_error_analysis2.items(), key = lambda x : x[1], reverse=True)))
print(f"Number of corrects : {new_nb_corrects2} || Target number of corrects : {final_correct}")
print(f"Total noise count : {new_total_noise_count2} || Target noise counts : {final_noise_count} || Total noise count if we add 'missing' errors : {sum(new_error_analysis2.values())}")
print("Total number of true spans :", total_n_true)

In [ ]:
ordered_error_start = {key : error_analysis[key] for key in final_target_numbers}
print("Starting errors : \n", ordered_error_start) 

print("target_numbers : \n", final_target_numbers)

ordered_error_end2 = {key : new_error_analysis2[key] for key in final_target_numbers}
print("Current errors : \n", ordered_error_end2)


In [ ]:
end_distribution2 = {key:new_error_analysis2[key]/np.sum(list(new_error_analysis2.values())) for key in target_distribution}
end_distribution2

# This is the format that needs to be respected for LLM few-shot settings (to save in data) 

In [ ]:
# For original scenario (most noise without any reduction)
# model_version2 = "BioLinkBERT-base_version=v1_noise=0.4"
# data_path2 = f"../data/{dataset_name}_{model_version2}.json"

# 0 noise
data_path2 = f"../data/{dataset_name}/{dataset_name}.json"

with open(data_path2, "r") as f:
    data2 = json.load(f)

In [ ]:
corrupted_dataset = {}
# for res in full_results: # For saving results with original noise
for res in new_full_results: # !! For saving noise reduced results with 1 pass !! 
# for res in new_full_results2: # !! For saving noise reduced results with 2 passes !!
    new_abstract = create_tagged_text(data_train_v1[res['pmid']]['input'], res['pred_spans'])
    pmid = res['pmid']
    corrupted_dataset[pmid] = {
        "modified": True,
        "input": data_train_v1[pmid]["input"],
        "output": new_abstract,
        "spans": res['pred_spans'],
        "split": "train",
    }
print(len(corrupted_dataset))

In [ ]:
corrupted_dataset_pmids = [key for key, item in corrupted_dataset.items()]

for pmid in data2.keys() :
    if pmid not in corrupted_dataset:
        corrupted_dataset[pmid] = data2[pmid]
len(corrupted_dataset) 

In [ ]:
# For noise-reduction scenario
data_path = f"../data/{dataset_name}/{dataset_name}_{model_version}_noise={wanted_noise}_dist={distribution}_{version}.json"

## For original scenario (most noise without any reduction)
# model_version = "phi-4_v16"
# model_version = "Llama-3.1-8B-Instruct_v4"
# model_version = "Mistral-Nemo-Instruct-2407_v5"
# model_version = "gemma-3-12b-it_dynamic=True_k=5_v2"
# model_version = "Mistral-Small-3.1-24B-Instruct-2503_dynamic=True_k=5_v3"
# data_path = f"../data/{dataset_name}/{dataset_name}_{model_version}.json"

with open(data_path, "w") as f:
    json.dump(corrupted_dataset, f, indent=4)

data_path

In [ ]:
# wanted_noise = "max"
# wanted_noise = 1
data_path = f"../data/{dataset_name}/{dataset_name}_{model_version}_noise={wanted_noise}_dist={distribution}_{version}.json"
data_path

## Sanity check

In [ ]:
final_noise = {}
nb_invalids_noise = set()
nb_invalids_empty = set()
nb_invalids_hallucination = set()
nb_untouched = set()
nb_corrects = 0
total_noise_count = 0
total_correct_count = 0
# total_n_true = 0

full_results = []

error_analysis = defaultdict(int)


for pmid in corrupted_dataset:
    if pmid not in data_train_v1:
        continue # Only on training data
    llm_annotation = corrupted_dataset[pmid]["output"]
    abstract, true_spans = data[pmid]['input'], data_train_v1[pmid]['spans']
    # true_spans = fuse_adjacent_tag(true_spans)
    pred_spans, reconstructed_text, count = gather_tagged_entities(
        abstract, llm_annotation, tag2label_st21pv
    ) 
    pred_spans = [p for p in pred_spans if (p["start"] != -1 and (p["end"]-p["start"]<70))]
    pred_spans = sorted(pred_spans, key=lambda x: x['start'])
    # pred_spans = fuse_adjacent_tag(pred_spans)
    f1, precision, recall, correct_count, noise_count, results, noise_types = compute_noise(pred_spans, true_spans)
    noise = 1-f1
    if reconstructed_text != abstract: # 1) if reconstructed text is not the same as the abstract, skip because the corruption really likely went wrong.
        nb_invalids_hallucination.add(pmid)
        continue
    if len(pred_spans) == 0: # 2) if pred_spans have less than 2 annotations, skip because no minimum of annotations per abstract in the dataset is 4.
        nb_invalids_empty.add(pmid)
        continue
    total_correct_count += correct_count
    total_noise_count += noise_count # Add noise only if it's not among the previous errors 
    # total_n_true += len(true_spans)
    final_noise[pmid] = noise
    for span in results:
        if span['correct'] == False:
            error_analysis[span['error_type']] += 1
        else :
            nb_corrects += 1
    full_results.append({"noise" : noise,
                         "f1" :f1, 
                         "precision" : precision,
                         "recall" : recall,
                         "correct_count" : correct_count,
                         "noise_count" : noise_count, 
                         "tagged_text" : llm_annotation,
                         "pred_spans" : pred_spans,
                         "true_spans" : true_spans,
                         "results" : results,
                         "noise_types" : noise_types,
                         "pmid" : pmid})

avg_noise = np.mean(list(final_noise.values())) * len(final_noise) / len(data_train_v1)
final_precision = total_correct_count / (total_correct_count + total_noise_count)
final_recall = total_correct_count / total_n_true
final_f1 = 2 * final_precision * final_recall / (final_precision + final_recall)
normalized_noise = 1-final_f1
print("Noise (each dataset contributed equally):", avg_noise) # Noise over the whole dataset
print("Normalized Noise (Dataset with more tags contributes more):", normalized_noise) # Noise over the whole dataset
print(f"Number of invalid corrupted abstracts : \n len(pred_spans) < 3 : {len(nb_invalids_empty)} || Reconstructed != original : {len(nb_invalids_hallucination)} || Untouched abstract : {len(nb_untouched)}") # || Formatting issue : {len(raw_noisy_data)-len(noisy_data)} \n")
print("Error per type :", dict(sorted(error_analysis.items(), key = lambda x : x[1], reverse=True)))
print("Number of corrects :", nb_corrects)
print("Total noise count :", total_noise_count, "|| Total noise count if we add 'missing' errors :", sum(error_analysis.values())) # Total noise count including all errors
print("Total number of true spans :", total_n_true)

# This is the format that needs to be respected for BERT-base model training (to save in data2)

In [ ]:
corrupted_dataset = []
# for res in full_results: # !! For saving results with original noise !!
for res in new_full_results: # !! For saving noise reduced results with 1 pass !! 
# for res in new_full_results2: # !! For saving noise reduced results with 2 passes !! 
    pmid = res['pmid']
    corrupted_dataset.append({
        "modified": True,
        "text": data_train_v1[pmid]["input"],
        "spans": res['pred_spans'],
        "split": "train",
        "pmid": pmid ,
    })
    # break
# assert len(corrupted_dataset) == len(new_full_results)
# assert len(corrupted_dataset) == len(full_results3)
print(f"There are {len(data_train) - len(corrupted_dataset)} that were not corrupted out of {len(data_train)}")

In [ ]:
path_to_original_data = f"../data2/{dataset_name}.json"
with open(path_to_original_data, "r") as f:
    original_data = json.load(f)
    
corrupted_dataset_pmids = [corrupted_dataset[i]['pmid'] for i in range(len(corrupted_dataset))]

for d in original_data:
    if d['pmid'] not in corrupted_dataset_pmids:
        corrupted_dataset.append(d)

assert len(corrupted_dataset) == len(original_data), \
    f"Number of corrupted dataset {len(corrupted_dataset)} is not equal to the original dataset {len(original_data)}"

In [ ]:
data_path = f"../data2/{dataset_name}_{model_version}_noise={wanted_noise}_dist={distribution}_{version}.json"
# data_path = f"../data2/{dataset_name}_{model_version}.json"
with open(data_path, "w") as f:
    json.dump(corrupted_dataset, f, indent=4)
data_path

In [ ]:
# data_path = f"../data2/{dataset_name}_{model_version}.json"

wanted_noise = "max"
data_path = f"../data2/{dataset_name}_{model_version}_noise={wanted_noise}_dist={distribution}_{version}.json"
data_path

# 23 Sept 2025

In [12]:
# dataset_name = "mm_st21pv"
# dataset_dir = "mm_st21pv"
# tag_to_label = tag2label_st21pv
# extraction_prompts = extraction_prompts_st21pv_v2

dataset_name = "bc5cdr"
dataset_dir = "bc5cdr"
tag_to_label = tag2label_bc5cdr
extraction_prompts = extraction_prompts_bc5cdr

# dataset_name = "ncbi_disease"
# dataset_dir = "ncbi_disease"
# tag_to_label = tag2label_ncbi
# extraction_prompts = extraction_prompts_ncbi

data_path = f"../data/{dataset_dir}/{dataset_name}.json"
with open(data_path, "r") as f:
    data = json.load(f)

for pmid, d in data.items() : 
    d['pmid'] = pmid
data_train = [data[pmid] for pmid in data if data[pmid]["split"] == "train"]

print(len(data), len(data_train))

1500 500


In [ ]:
data_folder = "mm_st21pv"

# model_version = "phi-4_v16"
# model_version = "phi-4_v15_noise=0.7"
# model_version = "gpt-4o-2024-11-20_noise=0.2"
# model_version = "BioLinkBERT-base_v1_noise=0.4"
# model_version = "BioLinkBERT-base_v1_noise=0.1_v2"
# model_version = "BioLinkBERT-base_train_v1"
model_version = "BioLinkBERT-base_test_v1"
# model_version = "BioLinkBERT-base_train_v1_noise=0.5"
# model_version = "Mistral-Nemo-Instruct-2407_v5"
# model_version = "Llama-3.1-8B-Instruct_v2"
# model_version = "gpt-4o-2024-11-20_dynamic=True_k=10_v11_noise=0.6"
# model_version = "gemma-3-12b-it_v2"
# model_version = "gemma-3-12b-it_v2_noise=0.8"
# model_version = "gemma-3-12b-it_v2_noise=0.1496_dist=wl"
# model_version = "Mistral-Small-3.1-24B-Instruct-2503_dynamic=True_k=5_v3"

noisy_data_path = f"../data/{data_folder}/{dataset_name}_{model_version}.json"
# noisy_data_path = f"/home/cye73/biomedicalNER/src/data/mm_st21pv/BioLinkBERT-base_version=v1.json"
with open(noisy_data_path, "r") as f:
    noisy_data = json.load(f)
print("Number of noisy data :", len(noisy_data))

noisy_data = {pmid: noisy_data[pmid] for pmid in noisy_data if noisy_data[pmid]['split']=='test'} 
data_train_v1 = {pmid: data[pmid] for pmid in data if data[pmid]["split"] == "test"}

total_n_true = 0
for pmid in noisy_data:
    true_spans = data_train_v1[pmid]['spans']
    # true_spans = fuse_adjacent_tag(true_spans)
    total_n_true += len(true_spans)

Number of noisy data : 4392


In [89]:
len(raw_noisy_data), len(noisy_data)

(40, 2635)

In [17]:
final_noise = {}
nb_invalids_noise = set()
nb_invalids_empty = set()
nb_invalids_hallucination = set()
nb_untouched = set()
nb_corrects = 0
total_noise_count = 0
total_correct_count = 0
total_n_true = 0

full_results = []

error_analysis = defaultdict(int)

for pmid in noisy_data:
    llm_annotation = noisy_data[pmid]["output"]
    abstract, true_spans = data[pmid]['input'], data_train_v1[pmid]['spans']
    pred_spans = noisy_data[pmid]['spans']
    
    f1, precision, recall, correct_count, noise_count, results, noise_types = compute_noise(pred_spans, true_spans)
    noise = 1-f1
    total_correct_count += correct_count
    total_noise_count += noise_count # Add noise only if it's not among the previous errors 
    total_n_true += len(true_spans)
    final_noise[pmid] = noise
    for span in results:
        if span['correct'] == False:
            error_analysis[span['error_type']] += 1
        else :
            nb_corrects += 1
    full_results.append({"noise" : noise,
                         "f1" :f1, 
                         "precision" : precision,
                         "recall" : recall,
                         "correct_count" : correct_count,
                         "noise_count" : noise_count, 
                         "tagged_text" : llm_annotation,
                         "pred_spans" : pred_spans,
                         "true_spans" : true_spans,
                         "results" : results,
                         "noise_types" : noise_types,
                         "pmid" : pmid})
    
avg_noise = np.mean(list(final_noise.values())) * len(final_noise) / len(data_train_v1)
final_precision = total_correct_count / (total_correct_count + total_noise_count)
final_recall = total_correct_count / total_n_true
final_f1 = 2 * final_precision * final_recall / (final_precision + final_recall)
normalized_noise = 1-final_f1
print(f"final_f1 : {final_f1} | final_precision : {final_precision} | final_recall : {final_recall}")
print("Noise (each dataset contributed equally):", avg_noise) # Noise over the whole dataset
print("Normalized Noise (Dataset with more tags contributes more):", normalized_noise) # Noise over the whole dataset
print(f"Number of invalid corrupted abstracts : \n len(pred_spans)==0 : {len(nb_invalids_empty)} || Reconstructed != original : {len(nb_invalids_hallucination)} || Untouched abstract : {len(nb_untouched)} || Formatting issue : {len(raw_noisy_data)-len(noisy_data)} \n")
print("Error per type :", dict(sorted(error_analysis.items(), key = lambda x : x[1], reverse=True)))
print("Number of corrects :", nb_corrects)
print("Total noise count :", total_noise_count, "|| Total noise count if we add 'missing' errors :", sum(error_analysis.values()))
print("Total number of true spans :", total_n_true)

final_f1 : 0.5626744703518402 | final_precision : 0.6134078212290502 | final_recall : 0.5196921007398551
Noise (each dataset contributed equally): 0.4551050805969417
Normalized Noise (Dataset with more tags contributes more): 0.43732552964815985
Number of invalid corrupted abstracts : 
 len(pred_spans)==0 : 0 || Reconstructed != original : 0 || Untouched abstract : 0 || Formatting issue : -839 

Error per type : {'missing': 11417, 'wrong_overlap': 5427, 'invalid_tag': 4617, 'wrong_label': 2159, 'multiple_entities': 945}
Number of corrects : 20862
Total noise count : 13148 || Total noise count if we add 'missing' errors : 24565
Total number of true spans : 40143


In [13]:
## LLM results from dataset corruption

data_folder = "mm_st21pv"
data_folder = dataset_name
# model_version = "gpt-4o-2024-11-20_dynamic=True_k=10_testSet_v6"
# model_version = "mistral-small-3.1-24b-instruct-2503-hf_dynamic=True_k=10_testSet_v4"
model_version = "Llama-3.1-8B-Instruct_dynamic=True_k=10_testSet_v5"
# data_folder = "mm_st21pv_BioLinkBERT_noise_50pc"
model_version = 'gemma-3-12b-it_dynamic=True_k=10_testSet_v5'

# data_folder = "mm_st21pv_gemma-3-12b-it_noise_50pc"
# data_folder = "mm_st21pv_gpt4o_noise_50pc"
# model_version = "Mistral-Small-3.1-24B-Instruct-2503_dynamic=True_k=10_v1"


# model_version = "gpt-4o-2024-11-20_version=v4"
# model_version = "phi-4_version=v15"
# model_version = "Llama-3.1-8B-Instruct_version=v4"
# model_version = "Qwen2.5-14B-Instruct_version=v6"
# model_version = "Mistral-Nemo-Instruct-2407_version=v5"

## LLM results for annotations
# model_version = "gpt-4o-2024-11-20_dynamic=True_k=10_v11"
# model_version = "gpt-4o-2024-11-20_dynamic=True_k=10_testSet"
# model_version = "gpt-4o-2024-11-20_dynamic=True_k=20_version=v1"
# model_version = "phi-4_dynamic=True_k=5_version=v11"
# model_version = "phi-4_dynamic=True_k=5_testSet_v4"
# model_version = "Llama-3.1-8B-Instruct_dynamic=True_k=3_version=v1"
# model_version = "Llama-3.1-8B-Instruct_dynamic=True_k=1_testSet_v4"
# model_version = "Mistral-Nemo-Instruct-2407_dynamic=True_k=3_version=v1"
# model_version = "gpt-4o-2024-11-20_dynamic=True_k=5_version=v1"
# model_version = "gemma-3-12b-it_dynamic=True_k=5_version=v1"
# model_version = "gemma-3-12b-it_dynamic=True_k=10_testSet_v4"
# model_version = "Mistral-Small-3.1-24B-Instruct-2503_dynamic=True_k=3_v1"
# model_version = "Mistral-Small-3.1-24B-Instruct-2503_dynamic=True_k=5_v2"
# model_version = "mistral-small-3.1-24b-instruct-2503-hf_dynamic=True_k=10_testSet_v4"


# data_folder = "bc5cdr"
# data_folder = "ncbi_disease"
# model_version = "Llama-3.1-8B-Instruct_dynamic=True_k=10_testSet_v2"
# model_version = "gemma-3-27b-it_dynamic=True_k=10_testSet_v2"
# model_version = "gpt-4o-2024-11-20_dynamic=True_k=10_testSet_v2"
# model_version = "phi-4_dynamic=True_k=10_testSet_v2"
# model_version = "mistral-small-3.1-24b-instruct-2503-hf_dynamic=True_k=10_testSet_v2"

raw_noisy_data_path = f"../llm/raw_results/{data_folder}/{model_version}.json" # Results from LLM inference
# raw_noisy_data_path = f"../raw_noisy_datasets/{data_folder}/{model_version}.json" # Results from dataset corruption
with open(raw_noisy_data_path, "r") as f:
    raw_noisy_data = json.load(f)
print("Number of raw noisy data :", len(raw_noisy_data))

noisy_data_path = f"../llm/results/{data_folder}/{model_version}.json" # Results from LLM inference
# noisy_data_path = f"../noisy_datasets/{data_folder}/{model_version}.json" # Results from dataset corruption
with open(noisy_data_path, "r") as f:
    llm_output = json.load(f)
with open(noisy_data_path, "r") as f:
    noisy_data = json.load(f)

print("Number of noisy data :", len(noisy_data))

# data_train_v1 = {pmid: data[pmid] for pmid in data if data[pmid]["split"] == "test"} # For LLM inference results
# data_train_v1 = {pmid: data[pmid] for pmid in data if data[pmid]["split"] == "train"} # For dataset corruption results
# total_n_true = sum([len(data[pmid]['spans']) for pmid in data_train_v1])
# noisy_data = {d['pmid']: d for d in noisy_data} # (not needed for results that were merged, they are already in a dict format)

noisy_data = {item["pmid"]: item for item in noisy_data} 
data_train_v1 = {pmid: data[pmid] for pmid in noisy_data}

total_n_true = 0
for pmid in noisy_data:
    true_spans = data_train_v1[pmid]['spans']
    true_spans = fuse_adjacent_tag(true_spans)
    total_n_true += len(true_spans)

# raw_noisy_data = {d['pmid']: d for d in raw_noisy_data} #

Number of raw noisy data : 500
Number of noisy data : 500


In [14]:
final_noise = {}
nb_invalids_noise = set()
nb_invalids_empty = set()
nb_invalids_hallucination = set()
nb_untouched = set()
nb_corrects = 0
total_noise_count = 0
total_correct_count = 0
total_n_true = 0

full_results = []

error_analysis = defaultdict(int)

for pmid in noisy_data:
# for pmid in list(noisy_data.keys())[:2]:
# for pmid in ['25763772']:
    llm_annotation = noisy_data[pmid]["tagged_text"]
    abstract, true_spans = data[pmid]['input'], data_train_v1[pmid]['spans']
    pred_spans = noisy_data[pmid]['pred_spans']
    
    # true_spans = fuse_adjacent_tag(true_spans)
    # pred_spans, reconstructed_text, count = gather_tagged_entities(
    #     abstract, llm_annotation, tag2label_st21pv
    # )
    # pred_spans = fuse_adjacent_tag(pred_spans)  
    # pred_spans = sorted(noisy_data[pmid]['pred_spans'], key=lambda x: x['start'])
    pred_spans = [p for p in pred_spans if p["start"] != -1]
    pred_spans = sorted(pred_spans, key=lambda x: x['start'])
    f1, precision, recall, correct_count, noise_count, results, noise_types = compute_noise(pred_spans, true_spans)
    noise = 1-f1
    # if reconstructed_text != abstract: # 1) if reconstructed text is not the same as the abstract, skip because the corruption really likely went wrong.
    #     nb_invalids_hallucination.add(pmid)
    #     continue
    if len(pred_spans) == 0: # 2) if pred_spans have less than 2 annotations, skip because no minimum of annotations per abstract in the dataset is 4.
        nb_invalids_empty.add(pmid)
        continue
    total_correct_count += correct_count
    total_noise_count += noise_count # Add noise only if it's not among the previous errors 
    total_n_true += len(true_spans)
    final_noise[pmid] = noise
    for span in results:
        if span['correct'] == False:
            error_analysis[span['error_type']] += 1
        else :
            nb_corrects += 1
    full_results.append({"noise" : noise,
                         "f1" :f1, 
                         "precision" : precision,
                         "recall" : recall,
                         "correct_count" : correct_count,
                         "noise_count" : noise_count, 
                         "tagged_text" : llm_annotation,
                         "pred_spans" : pred_spans,
                         "true_spans" : true_spans,
                         "results" : results,
                         "noise_types" : noise_types,
                         "pmid" : pmid})
    
avg_noise = np.mean(list(final_noise.values())) * len(final_noise) / len(data_train_v1)
final_precision = total_correct_count / (total_correct_count + total_noise_count)
final_recall = total_correct_count / total_n_true
final_f1 = 2 * final_precision * final_recall / (final_precision + final_recall)
normalized_noise = 1-final_f1
print(f"final_f1 : {final_f1} | final_precision : {final_precision} | final_recall : {final_recall}")
print("Noise (each dataset contributed equally):", avg_noise) # Noise over the whole dataset
print("Normalized Noise (Dataset with more tags contributes more):", normalized_noise) # Noise over the whole dataset
print(f"Number of invalid corrupted abstracts : \n len(pred_spans)==0 : {len(nb_invalids_empty)} || Reconstructed != original : {len(nb_invalids_hallucination)} || Untouched abstract : {len(nb_untouched)} || Formatting issue : {len(raw_noisy_data)-len(noisy_data)} \n")
print("Error per type :", dict(sorted(error_analysis.items(), key = lambda x : x[1], reverse=True)))
print("Number of corrects :", nb_corrects)
print("Total noise count :", total_noise_count, "|| Total noise count if we add 'missing' errors :", sum(error_analysis.values()))
print("Total number of true spans :", total_n_true)

final_f1 : 0.7169473507525318 | final_precision : 0.7671216475095786 | final_recall : 0.6729335153870392
Noise (each dataset contributed equally): 0.292346841133812
Normalized Noise (Dataset with more tags contributes more): 0.2830526492474682
Number of invalid corrupted abstracts : 
 len(pred_spans)==0 : 15 || Reconstructed != original : 0 || Untouched abstract : 0 || Formatting issue : 0 

Error per type : {'missing': 2504, 'invalid_tag': 1405, 'wrong_overlap': 474, 'multiple_entities': 58, 'wrong_label': 8}
Number of corrects : 6407
Total noise count : 1945 || Total noise count if we add 'missing' errors : 4449
Total number of true spans : 9521


In [15]:
nb_invalids_empty

{'11915580',
 '12359538',
 '15229250',
 '17346443',
 '17543491',
 '17879100',
 '18422462',
 '23892921',
 '24067251',
 '24158386',
 '25054547',
 '3711722',
 '573555',
 '8808730',
 '9158667'}

In [20]:
invalid_outputs = {pmid: noisy_data[pmid] for pmid in nb_invalids_empty}
pmid2verify ='3711722'
invalid_outputs[pmid2verify]

{'pmid': '3711722',
 'tagged_text': 'Human and canine ventricular vasoactive intestinal polypeptide: decrease with heart failure.\nVasoactive intestinal polypeptide (VIP) is a systemic and coronary vasodilator that may have positive inotropic properties. Myocardial levels of VIP were assayed before and after the development of heart failure in two canine models. In the first, cobalt cardiomyopathy was induced in eight dogs; VIP (by radioimmunoassay) decreased from 35 +/- 11 pg/mg protein (mean +/- SD) to 5 +/- 4 pg/mg protein (P less than 0.05). In six dogs with doxorubicin-induced heart failure, VIP decreased from 31 +/- 7 to 11 +/- 4 pg/mg protein (P less than 0.05). In addition, VIP content of left ventricular muscle of resected failing hearts in 10 patients receiving a heart transplant was compared with the papillary muscles in 14 patients (five with rheumatic disease, nine with myxomatous degeneration) receiving mitral valve prostheses. The lowest myocardial VIP concentration was 

In [15]:
invalid_outputs[pmid2verify]['tagged_text']
# noisy_data[pmid2verify]["tagged_text"]


'Eosinophilic Gastroenteritis as a Rare Cause of Recurrent Epigastric Pain\\nEosinophilic gastroenteritis (EGE) is a rare inflammatory disorder of gastrointestinal tract characterized by eosinophilic infiltration of the bowel wall. It can mimic many gastrointestinal disorders due to its wide spectrum of presentations. Diagnose is mostly based on excluding other disorders and a high suspicion. Here we report a case of 26 year old man with a history of sever epigastric pain followed by nausea, vomiting since a few days before admission with final diagnosis of EGE. \\n@@Eosinophilic gastroenteritis##T038@@ as a rare @@healthcare activity##T058@@ of recurrent @@epigastric pain##T033@@\\n@@Eosinophilic gastroenteritis##T038@@ (@@EGE##T038@@) is a rare @@inflammatory disorder##T038@@ of @@gastrointestinal tract##T017@@ characterized by @@eosinophilic infiltration##T038@@ of the @@bowel wall##T017@@. It can mimic many @@gastrointestinal disorders##T038@@ due to its wide spectrum of @@presenta

In [31]:
idx = 2
print("pmid : ", full_results[idx]['pmid'])
print("f1 : ", full_results[idx]['f1'], " | precision : ", full_results[idx]['precision'], " | recall : ", full_results[idx]['recall'])
print("correct_count : ", full_results[idx]['correct_count'], " | noise_count (doesn't consider 'missing' as noise) : ", full_results[idx]['noise_count'])
full_results[idx]['noise_types']

pmid :  27059693
f1 :  0.6000000000000001  | precision :  0.5675675675675675  | recall :  0.6363636363636364
correct_count :  21  | noise_count (doesn't consider 'missing' as noise) :  16


{'wrong_label': 5,
 'wrong_overlap': 3,
 'multiple_entities': 0,
 'invalid_tag': 8,
 'missing': 4}

In [19]:
full_results[idx]['results']

[{'start': 10,
  'end': 19,
  'label': 18,
  'tag': 'T097',
  'text': 'Physician',
  'correct': True,
  'error_type': None},
 {'start': 35,
  'end': 52,
  'label': 0,
  'tag': 'T058',
  'text': 'Global Assessment',
  'correct': False,
  'error_type': 'invalid_tag'},
 {'start': 56,
  'end': 76,
  'label': 3,
  'tag': 'T038',
  'text': 'Rheumatoid Arthritis',
  'correct': True,
  'error_type': None},
 {'start': 80,
  'end': 108,
  'label': 20,
  'tag': 'T170',
  'text': 'Systematic Literature Review',
  'correct': False,
  'error_type': 'wrong_overlap'},
 {'start': 114,
  'end': 127,
  'label': 1,
  'tag': 'T062',
  'text': 'Meta-Analysis',
  'correct': True,
  'error_type': None},
 {'start': 174,
  'end': 189,
  'label': 3,
  'tag': 'T038',
  'text': 'decision-making',
  'correct': True,
  'error_type': None},
 {'start': 210,
  'end': 220,
  'label': 0,
  'tag': 'T058',
  'text': 'management',
  'correct': True,
  'error_type': None},
 {'start': 224,
  'end': 244,
  'label': 3,
  'tag':

In [20]:
full_results[idx]['tagged_text']

"Patient - @@Physician##T097@@ Discordance in @@Global Assessment##T058@@ in @@Rheumatoid Arthritis##T038@@: A @@Systematic Literature Review##T170@@ With @@Meta-Analysis##T062@@\nThe integration of the patient in therapeutic @@decision-making##T038@@ is important in the @@management##T058@@ of @@rheumatoid arthritis##T038@@ (@@RA##T038@@), but the patient opinion regarding disease status may differ from the @@physician's##T097@@ opinion. The aim of this study was to assess in the @@published literature##T170@@ the frequency and drivers of patient - @@physician##T097@@ discordance in @@global assessment##T058@@ in @@RA##T038@@. A @@systematic literature review##T170@@ of all @@articles##T170@@ published up to January 2015 in @@Medline##T170@@ or @@Embase##T170@@, reporting discordance in @@RA##T038@@, was conducted by 2 @@investigators##T097@@. Discordance was defined based on the absolute difference of @@patient global##T170@@ (@@PGA##T170@@) and @@physician global assessments##T170@@

# Mean and variance

In [16]:
def mean_and_variance(F1, P, R):
    
    # Mean
    mean_F1 = sum(F1) / len(F1)
    mean_P = sum(P) / len(P)
    mean_R = sum(R) / len(R)
    
    # Variance (population variance by default)
    variance_F1 = sum((x - mean_F1) ** 2 for x in F1) / len(F1)
    variance_P = sum((x - mean_P) ** 2 for x in P) / len(P)
    variance_R = sum((x - mean_R) ** 2 for x in R) / len(R)

    return (mean_F1, variance_F1), (mean_P, variance_P), (mean_R, variance_R)


# Example usage:
F1_scores = [0.02, 1 - 99/100, 1 - 99/100]
F1_scores = [0.649, 0.619, 0.619]
P_scores = [0.696, 0.700, 0.700]
R_scores = [0.608, 0.565, 0.565]

(mean_F1, variance_F1), (mean_P, variance_P), (mean_R, variance_R) = mean_and_variance(F1_scores, P_scores, R_scores)

print("Mean F1 :", mean_F1, "| Variance F1 :", variance_F1)
print("Mean P  :", mean_P, "| Variance P  :", variance_P)
print("Mean R  :", mean_R, "| Variance R  :", variance_R)


Mean F1 : 0.629 | Variance F1 : 0.00020000000000000033
Mean P  : 0.6986666666666667 | Variance P  : 3.555555555555562e-06
Mean R  : 0.5793333333333334 | Variance R  : 0.0004108888888888896


In [14]:
P = 0.685
R = 0.565
F1 = 2 * P * R / (P + R)
F1

0.61924

In [16]:
1 - 804/879

0.08532423208191131

# Test on PICO

In [6]:
from datasets import load_dataset
import pandas as pd

# Load samples from ebm_pico
ebm_pico = load_dataset("bigbio/ebm_pico", "ebm_pico_bigbio_kb", split="train")
# Load samples from pico_extraction
pico_extraction = load_dataset("bigbio/pico_extraction", "pico_extraction_bigbio_kb", split="train")

print("=== EBM PICO sample ===")
print(pd.DataFrame(ebm_pico))

print("\n=== PICO Extraction sample ===")
print(pd.DataFrame(pico_extraction))


=== EBM PICO sample ===
         id document_id                                           passages  \
0         0    10036953  [{'id': '1', 'type': 'document', 'text': ['[ T...   
1        26    10037531  [{'id': '27', 'type': 'document', 'text': ['Xy...   
2        30    10052279  [{'id': '31', 'type': 'document', 'text': ['Pr...   
3        57    10071998  [{'id': '58', 'type': 'document', 'text': ['An...   
4        77    10073522  [{'id': '78', 'type': 'document', 'text': ['Fe...   
...     ...         ...                                                ...   
4741  93723     9950438  [{'id': '93724', 'type': 'document', 'text': [...   
4742  93738     9951491  [{'id': '93739', 'type': 'document', 'text': [...   
4743  93755      998549  [{'id': '93756', 'type': 'document', 'text': [...   
4744  93778     9987969  [{'id': '93779', 'type': 'document', 'text': [...   
4745  93794     9989713  [{'id': '93795', 'type': 'document', 'text': [...   

                                       

In [8]:
# Extract all entity types
all_types = set()
for example in ebm_pico:
    for ent in example["entities"]:
        all_types.add(ent["type"])

print("Unique entity types:", all_types)
print("Number of unique types:", len(all_types))

Unique entity types: {'Intervention_Other', 'Participant_Condition', 'Intervention_Surgical', 'Intervention_Educational', 'Outcome_Pain', 'Intervention_Psychological', 'Outcome_Adverse-effects', 'Intervention_Physical', 'Outcome_Mortality', 'Participant_Sample-size', 'Outcome_Physical', 'Outcome_Mental', 'Outcome_Other', 'Participant_Sex', 'Intervention_Pharmacological', 'Intervention_Control', 'Participant_Age'}
Number of unique types: 17


In [9]:
# Extract all entity types
all_types = set()
for example in pico_extraction:
    for ent in example["entities"]:
        all_types.add(ent["type"])

print("Unique entity types:", all_types)
print("Number of unique types:", len(all_types))

Unique entity types: {'intervention', 'outcome', 'participant'}
Number of unique types: 3
